In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
pd.set_option("display.max_columns", None)
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


In [2]:
# Import Raw data and additional libraries
from sklearn.impute import SimpleImputer # handles missing values
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer # One hot encoder, transform custom functions to fit in Pipelines 
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer # transform datas in a specified column inside the Pipeline
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import mutual_info_regression
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from category_encoders import MEstimateEncoder
from sklearn.feature_selection import SelectKBest, mutual_info_classif

Raw_data = pd.read_csv("/kaggle/input/competitions/titanic/train.csv") # raw data csv
test_data = pd.read_csv("/kaggle/input/competitions/titanic/test.csv")

In [3]:
# view Raw data
Raw_data['Age'] = Raw_data['Age'].fillna(Raw_data['Age'].mean())
test_data['Age'] = test_data['Age'].fillna(test_data['Age'].mean())
# Create the column globally for your training data splits before fitting
global_scaler = StandardScaler()

for df in [Raw_data, test_data]:
    df['Fare'] = df['Fare'].fillna(Raw_data['Fare'].mean())
    df['Age'] = df['Age'].fillna(Raw_data['Age'].median()) # Handle Age NaN!
    
    ticket_counts = df['Ticket'].map(Raw_data['Ticket'].value_counts()).fillna(1)
    df['Average_Fare'] = df['Fare'] / ticket_counts
    
    scaled_features = global_scaler.fit_transform(df[['Average_Fare', 'Age']])
    df['Average_Fare_Scaled'] = scaled_features[:, 0]
    df['Age_Scaled'] = scaled_features[:, 1]
print(Raw_data)

     PassengerId  Survived  Pclass  \
0              1         0       3   
1              2         1       1   
2              3         1       3   
3              4         1       1   
4              5         0       3   
..           ...       ...     ...   
886          887         0       2   
887          888         1       1   
888          889         0       3   
889          890         1       1   
890          891         0       3   

                                                  Name     Sex        Age  \
0                              Braund, Mr. Owen Harris    male  22.000000   
1    Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.000000   
2                               Heikkinen, Miss. Laina  female  26.000000   
3         Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.000000   
4                             Allen, Mr. William Henry    male  35.000000   
..                                                 ...     ...        ...   
886 

In [4]:
# Target encoding
TE_X = Raw_data.copy()
TE_y = TE_X.pop('Survived')

TE_X['Cabin'] = TE_X['Cabin'].fillna('X')
TE_X['Cabin'] = TE_X['Cabin'].astype(str).str[0]
TE_X_encode = TE_X.sample(frac=0.25, random_state=0)
TE_y_encode = TE_y[TE_X_encode.index]
TE_X_pretrain = TE_X.drop(TE_X_encode.index)
TE_y_train = TE_y[TE_X_pretrain.index]
for i in ["Sex", 'SibSp', "Parch", 'Cabin']:
    TE_X_encode[f'{i}_encoded'] = TE_X_encode[i]
    TE_X_pretrain[f'{i}_encoded'] = TE_X_pretrain[i]

encoder = MEstimateEncoder(cols=["Sex_encoded", 'SibSp_encoded', "Parch_encoded", 'Cabin_encoded'], m=5.0)
encoder.fit(TE_X_encode, TE_y_encode)
TE_X_train = encoder.transform(TE_X_pretrain)

In [5]:
# Base feature engineering
def cabin_letter_coding(data: pd.DataFrame) -> pd.DataFrame:
    data = data.copy()
    data['Cabin'] = data['Cabin'].fillna('X')
    data['Cabin'] = data['Cabin'].astype(str).str[0]
    return data[['Cabin']]

# new features after mi_score
def age_splitting(data: pd.DataFrame) -> pd.DataFrame:
    data = data.copy()
    bin_edges = [0, 5, 12, 18, 35, 60, np.inf]
    bin_labels = ['Toddler', 'Child', 'Teen', 'Young Adult', 'Adult', 'Senior']
    data['Age_group'] = pd.cut(data['Age'], bins=bin_edges, labels=bin_labels)
    data['Age_group'] = data['Age_group'].cat.codes
    return data[['Age_group']]

def count_ticket(data: pd.DataFrame) -> pd.DataFrame:
    data = data.copy()
    data['Ticket_Count'] = data['Ticket'].map(Raw_data['Ticket'].value_counts())
    return data[['Ticket_Count']]

def Surname_extractor(data: pd.DataFrame) -> pd.DataFrame:
    data = data.copy()
    data['Surname'] = data['Name'].str.split(',', expand=True)[0]
    data['Surname_count'] = data['Surname'].map(data['Surname'].value_counts())
    return data[['Surname_count']]
    
#data clustering with kmean
'''
def age_fare_class_cluster(data: pd.DataFrame) -> pd.DataFrame:
    data = data.copy()
    data['Age'] = data['Age'].fillna(data['Age'].mean())
    cluster_features = ['Age', 'Fare', 'Pclass']
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(data[cluster_features])
    kmeans = KMeans(n_clusters=15, random_state=0, n_init=15)
    data['Passenger_Cluster'] = kmeans.fit_predict(X_scaled)

    cluster_distance = kmeans.fit_transform(X_scaled)
    for i in range(15):
        data[f'Distance_to_Cluster_{i}'] = cluster_distance[:, i]
    return data[[f'Distance_to_Cluster_{i}' for i in range(15)] + ['Passenger_Cluster']]
'''

def average_fare(data: pd.DataFrame) -> pd.DataFrame:
    data = data.copy()
    data['Fare'] = data['Fare'].fillna(data['Fare'].mean())
    ticket_counts = data['Ticket'].map(Raw_data['Ticket'].value_counts())
    data["Average_Fare"] = data['Fare']/ticket_counts
    return data[["Average_Fare"]]
    
# applying PCA
'''
def pca_applyer(data: pd.DataFrame) -> pd.DataFrame:
    data = data.copy()
    pca = PCA()
    data['Fare'] = data['Fare'].fillna(data['Fare'].mean())
    ticket_counts = data['Ticket'].map(Raw_data['Ticket'].value_counts())
    data["Average_Fare"] = data['Fare']/ticket_counts
    X = data[['Average_Fare', 'Age']]
    X_pca = pca.fit_transform(X)
    component_names = [f"PC{i+1}" for i in range(X_pca.shape[1])]
    pca_col = pd.DataFrame(X_pca, index=data.index, columns=component_names)
    data = data.join(pca_col)
    return data[component_names]
'''
pca = PCA()
data = Raw_data.copy()
data['Fare'] = data['Fare'].fillna(data['Fare'].mean())
ticket_counts = data['Ticket'].map(Raw_data['Ticket'].value_counts())
data["Average_Fare"] = data['Fare']/ticket_counts
X = data[['Average_Fare', 'Age']]
X_scaled = (X - X.mean(axis=0)) / X.std(axis=0)
X_pca = pca.fit_transform(X_scaled)
loadings = pd.DataFrame(
    pca.components_.T,  # transpose the matrix of loadings
    columns=['PC0', 'PC1'],  # so the columns are the principal components
    index=['Average_Fare', 'Age'],  # and the rows are the original features
)
print(loadings)

gender_pipe = Pipeline([
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])
age_pipe = Pipeline([
    ('cat_age', FunctionTransformer(age_splitting)),
    ('onehot', OneHotEncoder())
])
fare_pipe = Pipeline([
    ('average_fare_cal', FunctionTransformer(average_fare))
])
cabin_pipe = Pipeline([
    ('extract_letter', FunctionTransformer(cabin_letter_coding)),
    ('onehot', OneHotEncoder())
])
ticket_pipe = Pipeline([
    ('count_tickets', FunctionTransformer(count_ticket))
])
surname_pipe = Pipeline([
    ('surname_extraction', FunctionTransformer(Surname_extractor))
])
cluster_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')), # 1. Fixes the NaN crash!
    ('scaler', StandardScaler()),                  # 2. Scales variables for KMeans geometry
    ('kmeans', KMeans(n_clusters=15, random_state=0, n_init=15)) # 3. Finds clusters
])
pca_pipe = Pipeline([
    ('pca', PCA(random_state=0))   # Learns PCA on train, applies to test
])
Base_preprocessor = ColumnTransformer([
    ('gender', gender_pipe, ['Sex']),
    ('age', age_pipe, ['Age']),
    ('fare', fare_pipe, ['Fare', "Ticket"]),
    ('cabin', cabin_pipe, ['Cabin']),
    ('pass_through', 'passthrough', ['Pclass', 'SibSp', 'Parch'])
])
preprocessor = ColumnTransformer([
    ('gender', gender_pipe, ['Sex']),
    ('age', age_pipe, ['Age']),
    ('fare', fare_pipe, ['Fare', "Ticket"]),
    ('cabin', cabin_pipe, ['Cabin']),
    ('ticket', ticket_pipe, ['Ticket']),
    ('name', surname_pipe, ['Name']),
    ('cluster', cluster_pipe, ['Age', 'Fare', 'Pclass']),
    ('pca', pca_pipe, ['Average_Fare', 'Age']),
    ('pass_through', 'passthrough', ['Pclass', 'SibSp', 'Parch', 'Age', "Sex_encoded", 'SibSp_encoded', "Parch_encoded", 'Cabin_encoded'])
])
Base_model = Pipeline([
    ('preprocessor', Base_preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=0, max_depth=5, max_features='sqrt', class_weight=None, min_samples_leaf=5))
])

                   PC0       PC1
Average_Fare  0.707107 -0.707107
Age           0.707107  0.707107


In [6]:
# Set base precision
Base_X = Raw_data[["Ticket", "Pclass", 'Sex', 'Age', 'Fare', 'Cabin', 'SibSp', 'Parch']]
Base_y = Raw_data["Survived"]
B_train_X, B_valid_X, B_train_y, B_valid_y = train_test_split(Base_X, Base_y, test_size=0.15, random_state=0)
Base_model.fit(B_train_X, B_train_y)
Base_prediction_results = Base_model.predict(B_valid_X)
accuracy = accuracy_score(B_valid_y, Base_prediction_results)
print(f"Accuracy: {accuracy * 100:.4f}%")

Accuracy: 80.5970%


In [7]:
# Adjusting the model to XGBoost
X = TE_X_train
y = TE_y_train
train_X, valid_X, train_y, valid_y = train_test_split(X, y, test_size=0.15, random_state=0)
    
preprocessor.fit(train_X)
transformed_X = preprocessor.transform(valid_X)

transformed_X_train = preprocessor.transform(train_X)

XGBoost_model = Pipeline([
    ('preprocessor', preprocessor),
    #('final_imputer', SimpleImputer(strategy='median')), # catch for any remaining NaNs
    #('feature_selection', SelectKBest(score_func=mutual_info_classif, k=45)),
    ('classifier', XGBClassifier(n_estimators=1500, learning_rate=0.02, early_stopping_rounds=25, random_state=0, colsample_bytree=0.6))
])
XGBoost_model.fit(
    train_X, 
    train_y,
    classifier__eval_set=[(transformed_X, valid_y)],
    classifier__verbose=False
)
XGBoost_prediction_results = XGBoost_model.predict(valid_X)
accuracy = accuracy_score(valid_y, XGBoost_prediction_results)
print(f"Accuracy: {accuracy * 100:.4f}%")

Accuracy: 83.1683%


In [8]:
# Using Mutual Imformation to analyse relation before deeper feature engineering
def calculate_mi_score(X, y, discrete_features):
    mi_scores = mutual_info_regression(X, y, random_state=0)
    mi_scores = pd.Series(mi_scores, name="MI scores", index=X.columns)
    mi_scores = mi_scores.sort_values(ascending=False)
    return mi_scores

gender_feature_names = preprocessor.named_transformers_['gender'].get_feature_names_out(['Sex'])
Cabin_feature_names = preprocessor.named_transformers_['cabin'].named_steps['onehot'].get_feature_names_out(['Cabin'])
age_feature_names = preprocessor.named_transformers_['age'].named_steps['onehot'].get_feature_names_out(['Age_group'])
feature_names = list(gender_feature_names) + list(age_feature_names) + ['Average_Fare'] + list(Cabin_feature_names) + ['Ticket_count', 'Surname_count'] + [f'Distance_to_Cluster_{i}' for i in range(15)] + [f"PC{i}" for i in range(2)] + ['Pclass', 'SibSp', 'Parch', 'Age'] + ["Sex_encoded", 'SibSp_encoded', "Parch_encoded", 'Cabin_encoded']
discrete_features = list(gender_feature_names) + list(age_feature_names) + list(Cabin_feature_names) + ['Passenger_Cluster'] + ['Ticket_count', 'Surname_count', 'Pclass', 'SibSp', 'Parch']

transformed_X_train = preprocessor.transform(train_X)
print("--- ARRAY SHAPE CHECK ---")
print(f"Total columns in transformed array: {transformed_X_train.shape[1]}")
print(f"Total names in your manual list:   {len(feature_names)}")

print("\n--- INDIVIDUAL STEP SHAPES ---")
for name, transformer in preprocessor.named_transformers_.items():
    if name != 'remainder':
        # Safely transform a small slice of data to see how many columns it outputs
        step_output = transformer.transform(train_X[preprocessor.transformers_[[t[0] for t in preprocessor.transformers].index(name)][2]])
        print(f"Step '{name}' outputs shape: {step_output.shape}")
print('\n')

processed_data = pd.DataFrame(transformed_X_train, columns=feature_names)
print(processed_data)
print(calculate_mi_score(processed_data, train_y, discrete_features))

--- ARRAY SHAPE CHECK ---
Total columns in transformed array: 45
Total names in your manual list:   45

--- INDIVIDUAL STEP SHAPES ---
Step 'gender' outputs shape: (567, 2)
Step 'age' outputs shape: (567, 6)
Step 'fare' outputs shape: (567, 1)
Step 'cabin' outputs shape: (567, 9)
Step 'ticket' outputs shape: (567, 1)
Step 'name' outputs shape: (567, 1)
Step 'cluster' outputs shape: (567, 15)
Step 'pca' outputs shape: (567, 2)
Step 'pass_through' outputs shape: (567, 8)


     Sex_female  Sex_male  Age_group_0  Age_group_1  Age_group_2  Age_group_3  \
0           0.0       1.0          0.0          0.0          0.0          1.0   
1           1.0       0.0          0.0          0.0          1.0          0.0   
2           0.0       1.0          0.0          0.0          0.0          1.0   
3           0.0       1.0          0.0          0.0          0.0          1.0   
4           1.0       0.0          0.0          0.0          0.0          1.0   
..          ...       ...          ...

In [9]:
# Final output results
processed_test = test_data.copy()

processed_test['Cabin'] = processed_test['Cabin'].fillna('X')
processed_test['Cabin'] = processed_test['Cabin'].astype(str).str[0]

for i in ["Sex", 'SibSp', "Parch", 'Cabin']:
    processed_test[f'{i}_encoded'] = processed_test[i]

processed_test = encoder.transform(processed_test)

# 5. Hand the prepared data directly to your XGBoost pipeline
prediction_results = XGBoost_model.predict(processed_test)
final_df = pd.DataFrame({
    "PassengerId" : test_data['PassengerId'],
    "Survived" : prediction_results
})
final_results = final_df.set_index('PassengerId')
final_results.to_csv("/kaggle/working/submission.csv")